# OR breakout: reach 4% beyond the line by 3:55 ET

**Definition:** Same 30-minute OR and first confirmed breakout as the earlier study. First completed 1-, 2-, or 5-minute candle close strictly outside the OR; choose the first break of either direction per ticker/day/timeframe. A success is a subsequent one-minute high ≥ OR high × 1.04 (upper break), or subsequent one-minute low ≤ OR low × 0.96 (lower break), by the saved session-end window. This measures movement from the **OR boundary**, not from the breakout fill or candle close. The confirming candle itself is excluded. A later return to the OR does not disqualify the event.

Read the saved `breakout_events_all.csv` in chunks; no market CSV replay is required. Report distinct **ticker-days** and **tickers**, because one ticker can qualify on many days. The primary rows use the same first-event flag and session-end data coverage as the earlier OR path study. Compare upper/lower and 1/2/5-minute results separately and combined. Also show what portion reached 4% only *after* a return to the OR using the smaller Phase 1 chronological file, with that subset's different coverage labeled separately; the primary 4% totals do not depend on that optional split.

This is a price-path count, not an executable 4% target backtest: 1-minute OHLC cannot establish whether a stop would have occurred before the 4% level.

In [ ]:
from google.colab import drive
from pathlib import Path
from IPython.display import display
import pandas as pd, numpy as np
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive')
SRC=BASE/'OR_Breakout_Excursion_1m_2m_5m'/'breakout_events_all.csv'
OUT=BASE/'OR_4pct_Continuation'
OUT.mkdir(exist_ok=True)
if not SRC.exists():raise FileNotFoundError(SRC)
cols=['ticker','date','timeframe_min','side','horizon','first_event','extension_pct',
      'return_touch' if False else 'touched','boundary','signal_time']
# pandas usecols discards the placeholder; primary only needs these existing fields.
cols=[c for c in cols if c!='return_touch']
parts=[]
for block in pd.read_csv(SRC,usecols=cols,chunksize=200_000,
                         dtype={'ticker':str,'date':str,'horizon':str},low_memory=False):
    first=block.first_event.astype(str).str.lower().eq('true')
    v=block[first & block.horizon.eq('session_end')].copy()
    if not v.empty:parts.append(v)
if not parts:raise RuntimeError('No first-event session-end rows; check source file')
d=pd.concat(parts,ignore_index=True)
assert not d.duplicated(['ticker','date','timeframe_min']).any()
d['reached_4pct']=d.extension_pct.ge(4)
d['reached_2pct']=d.extension_pct.ge(2)
d['reached_3pct']=d.extension_pct.ge(3)
d.to_csv(OUT/'first_break_4pct_event_ledger.csv',index=False)
print('First breakout ticker-days across three alternative candle sizes:',len(d))
print('Distinct tickers in universe:',d.ticker.nunique(),'dates:',d.date.nunique())

In [ ]:
def summarize(g):
    winners=g[g.reached_4pct]
    return pd.Series({'breakout_ticker_days':len(g),
        'reached_4pct_ticker_days':len(winners),
        'reached_4pct_pct':100*len(winners)/len(g),
        'distinct_tickers_with_breakout':g.ticker.nunique(),
        'distinct_tickers_reaching_4pct':winners.ticker.nunique(),
        'dates':g.date.nunique(),
        'reached_2pct_ticker_days':int(g.reached_2pct.sum()),
        'reached_3pct_ticker_days':int(g.reached_3pct.sum())})
by_side=d.groupby(['timeframe_min','side'],dropna=False).apply(summarize,include_groups=False).reset_index()
combined=d.groupby(['timeframe_min'],dropna=False).apply(summarize,include_groups=False).reset_index()
by_side.to_csv(OUT/'4pct_by_timeframe_and_direction.csv',index=False)
combined.to_csv(OUT/'4pct_both_directions_unique_ticker_days.csv',index=False)
print('UPPER AND LOWER SEPARATELY');display(by_side.round(2))
print('BOTH DIRECTIONS — first break is one side per ticker-day/timeframe');display(combined.round(2))
# Month/date counts show whether a few market sessions dominate.
d['month']=d.date.str[:7]
for name,keys in [('month',['month','timeframe_min','side']),('date',['date','timeframe_min','side'])]:
    z=d.groupby(keys,dropna=False).apply(summarize,include_groups=False).reset_index()
    z.to_csv(OUT/('4pct_by_'+name+'.csv'),index=False)
    if name=='month':display(z.round(2))
print('Saved reports to',OUT)

In [ ]:
# Optional timing split on Phase 1's stricter complete-minute cohort.
# Distinguish 4% BEFORE first return vs later, without changing primary totals above.
CHRON=BASE/'OR_Phase1_Chronology_Stop_Target'/'phase1_chronological_paths.csv'
if CHRON.exists():
    c=pd.read_csv(CHRON,dtype={'ticker':str,'date':str,'horizon':str},low_memory=False)
    c=c[c.horizon.eq('session_end')].copy()
    c['4pct_before_first_return']=c.extension_before_return_pct.ge(4)
    c['4pct_after_first_return']=(
        100*c.extension_after_return_dollars/c.boundary).ge(4)
    c['4pct_anytime']=c['4pct_before_first_return']|c['4pct_after_first_return']
    timing=c.groupby(['timeframe_min','side']).agg(
        complete_minute_events=('ticker','size'),
        reached_before_return=('4pct_before_first_return','sum'),
        reached_after_return=('4pct_after_first_return','sum'),
        reached_anytime=('4pct_anytime','sum')).reset_index()
    timing.to_csv(OUT/'4pct_before_vs_after_return_strict_subset.csv',index=False)
    print('OPTIONAL STRICT COVERAGE SUBSET; denominators differ from primary:');display(timing)
print('Done.')